# Actividad 3: Aprendizaje supervisado y no supervisado

### Grupo: AE_Grupo_6
Integrantes:
* Juan Fernando Gaviria Higuita
* Juan David Ramírez García

## 0. Pregunta de negocio y construcción automática del DataFrame

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader

In [2]:
loader = HFDatasetLoader(
    dataset_name="infinite-dataset-hub/HealthVitalSigns",
    split="train",
    filename="datos/healthvitalSigns.csv",
)

In [3]:
df = loader.load()
df.head()

,idx,timestamp,heart_rate,blood_pressure,oxygen_saturation,body_temperature,label
0,0,2023-01-01T08:00:00Z,72,120/80,98,98.6,normal
1,1,2023-01-01T08:05:00Z,76,121/81,98,98.7,normal
2,2,2023-01-01T08:10:00Z,85,130/85,97,99.1,elevated
3,3,2023-01-01T08:15:00Z,95,135/89,96,100.4,critical
4,4,2023-01-01T08:20:00Z,70,115/75,99,98.4,normal


### Pregunta de Negocio

**¿Podemos predecir en tiempo real si un paciente requiere atención Normal, Elevada o Crítica analizando sus signos vitales?**

- **¿Quién se beneficia?**: El equipo médico y el personal de triaje en salas de urgencias u hospitales.

- **¿Qué decisión apoya?**: Clasificar y priorizar la atención de los pacientes según su gravedad. Permite asignar atención inmediata a quienes estén en estado Crítico, optimizar los recursos del hospital y reducir tiempos de espera en emergencias.

## Documentación del dataset

Enlace en huggingface del dataset:[HealthVitalSigns](https://huggingface.co/datasets/infinite-dataset-hub/HealthVitalSigns)

- **¿Qué datos tiene?**: Datos biométricos tomados a pacientes en intervalos de tiempo, orientados al monitoreo de salud.

- **Columnas relevantes**:

    - heart_rate: Ritmo cardíaco (latidos por minuto).
 
    - blood_pressure: Presión arterial expresada como texto (sistólica/diastólica).

    - oxygen_saturation: Porcentaje de saturación de oxígeno en sangre ($\text{SpO}_2$).

    - body_temperature: Temperatura corporal en grados Fahrenheit.label / risk_category:
 
    - Variable objetivo con la categoría de riesgo (normal, elevated, critical).
 
- **Justificación del Dataset**: El dataset HealthVitalSigns permite responder la pregunta porque relaciona directamente datos vitales como               heart_rate, blood_pressure, oxygen_saturation y body_temperature con el nivel de riesgo del paciente: normal, elevated y critical.

  Esto le permite al modelo aprender qué combinaciones de signos vitales corresponden a una urgencia médica y automatizar la priorización en triaje      sin comprometer datos de pacientes reales.

- **Variable Objetivo**: risk_category (normal, elevated, critical).

- **¿Cómo la obtuvimos?** : Tomamos la columna label que venía directamente en el dataset de Hugging Face y la renombrámos como risk_category para darle un nombre más claro dentro del contexto médico.

- **¿Por qué la elegimos?**: Porque indica la gravedad real del paciente. Da las respuesta que el modelo debe aprender a predecir para saber a quién atender primero en urgencias.

## Estructura dataFrame

In [4]:
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
print(f"Filas duplicadas: {df.duplicated().sum()}")
print(f"Valores nulos totales: {df.isnull().sum().sum()}\n")

print("--- Información de tipos de datos y memoria ---")
df.info()

Filas: 100
Columnas: 7
Filas duplicadas: 0
Valores nulos totales: 0

--- Información de tipos de datos y memoria ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   idx                100 non-null    int64  
 1   timestamp          100 non-null    object 
 2   heart_rate         100 non-null    int64  
 3   blood_pressure     100 non-null    object 
 4   oxygen_saturation  100 non-null    int64  
 5   body_temperature   100 non-null    float64
 6   label              100 non-null    object 
dtypes: float64(1), int64(3), object(3)
memory usage: 5.6+ KB


## 1. Evidencia de preprocesamiento

El dataFrame no tiene datos nulos entonces no haremos limpieza de nulos y de duplicados o imputación de datos.

In [5]:
# Eliminando la columna redundante 'idx' porque es la misma que el índice del DataFrame
df_clean= df.drop(columns=["idx"])
# Verificación de la lista de columnas
print(f"Verificando que la columna 'idx' ha sido eliminada:\n{df_clean.columns}")

Verificando que la columna 'idx' ha sido eliminada:
Index(['timestamp', 'heart_rate', 'blood_pressure', 'oxygen_saturation',
       'body_temperature', 'label'],
      dtype='object')


In [6]:
# Cambiando la columna 'label' por "risk_category
df_clean= df_clean.rename(columns={"label": "risk_category"})
print(f"Verficando cambio de nombre: \n{df_clean.columns}")

Verficando cambio de nombre: 
Index(['timestamp', 'heart_rate', 'blood_pressure', 'oxygen_saturation',
       'body_temperature', 'risk_category'],
      dtype='object')


In [7]:
print("=== COMPARACIÓN DE DATAFRAMES ===")
print(f"Dimensiones Original (df_raw): {df.shape}")
print(f"Dimensiones Limpio   (df_clean): {df_clean.shape}\n")

columnas_eliminadas = set(df.columns) - set(df_clean.columns)
columnas_agregadas = set(df_clean.columns) - set(df.columns)

print(f"Columnas eliminadas: {columnas_eliminadas}")
print(f"Columnas agregadas:   {columnas_agregadas}")

=== COMPARACIÓN DE DATAFRAMES ===
Dimensiones Original (df_raw): (100, 7)
Dimensiones Limpio   (df_clean): (100, 6)

Columnas eliminadas: {'label', 'idx'}
Columnas agregadas:   {'risk_category'}


**Justificación del Escalado**

- **No escalamos ni normalizamos los datos**: El algoritmo que elegimos es un Árbol de Decisión. A diferencia de otros modelos que miden distancias entre puntos en un mapa, el árbol de decisión solo se hace preguntas sencillas de tipo "es mayor que" o "es menor que" sobre cada constante vital por separado. Por ejemplo, no le afecta que la temperatura sea un número pequeño como 98 y el ritmo cardíaco sea un número más grande como 120; los evalúa de forma independiente sin importar sus unidades o tamaños.

- **Justificación de limpieza:**

    - Estado de los DatosNulos y duplicados: No se realizó imputación de datos, limpieza de nulos ni eliminación de duplicados debido a que el               DataFrame no presenta valores nulos.
    - Operaciones Ejecutadas en el CódigoEliminación de idx: Se borró la columna redundante 'idx' porque cumple la misma función que el índice del
       propio DataFrame.
    - Renombrado de la variable objetivo: Se cambió el nombre de la columna 'label' a 'risk_category'.   3. Comparación de DataFramesDimensiones del         DataFrame original (df_raw): (100, 7).
    -  Dimensiones del DataFrame limpio (df_clean): (100, 6).
    -  Columnas eliminadas: {'label', 'idx'}.
    -  Columnas agregadas: {'risk_category'}.
    -  Estructura final de columnas (df_clean): 'timestamp', 'heart_rate', 'blood_pressure', 'oxygen_saturation', 'body_temperature',                         'risk_category'.   

- **División de Entrenamiento y Prueba**


   - Separación de datos: Separamos el conjunto de datos en dos partes: un 80 por ciento para enseñarle al modelo (entrenamiento) y un 20 por ciento        para evaluar qué tan bien aprendió (prueba).

  - Control de aprendizaje: Para garantizar una evaluación real y sin sesgos, todas las transformaciones y reglas se aprenden únicamente del grupo de     entrenamiento. El grupo de prueba se mantiene guardado e intacto, y solo se le aplican las reglas aprendidas al final para verificar los resultados     reales.

## 2. Selección y funcionamiento del modelo

- **Tipo de aprendizaje y justificación**: Usamos un modelo de aprendizaje supervisado llamado Árbol de Decisión porque el conjunto de datos ya tenía las etiquetas de las respuestas a la pregunta del negocio. Entonces la idea es hacer que el modelo entrene con una porción de esos datos etiquetados para saber cómo clasificar a los nuevos pacientes que lleguen a urgencias.

- **Variables elegidas y justificación**: Nuestra variable objetivo es **risk_category** con etiquetas como normal, elevated, critical, que son justamente las eqtiquetas o repeustas de la pregutna de negocio que queremos predecir. Como variables predictoras usamos las columnas heart_rate, blood_pressure, oxygen_saturation y body_temperature porque son datos fisiológicos que nos dicen qué tan grave se encuentra una persona.

**Funcionamiento del modelo**

El árbol de decisión funciona respondiendo preguntas cerradas, en este caso preguntas relacionadas con nuestra pregunta de negocio. 
Por ejemplo, en la primera interacción, el algoritmo evalúa todos los signos vitales del paciente y selecciona cuál de ellos separa mejor a la población. Ejemplo: ¿su frecuencia cardíaca es mayor a 100 bpm? Entonces, respondiendo sí o no, divide a los pacientes en dos subgrupos: los que cumplen o no la condición. Luego, en las siguientes interacciones, repite exactamente el mismo proceso dentro de cada rama analizando el resto de las variables como la presión arterial o la temperatura para seguir refinando la clasificación. Finalmente, este flujo de preguntas termina en las "hojas" del árbol, donde se asigna el diagnóstico de riesgo definitivo del paciente si es normal, elevated o critical.

## 3. Entrenamiento, métricas y evaluación

In [8]:
from clases_po import ArbolDecision

# 1. Instanciar la clase
arbol = ArbolDecision(max_depth=4, random_state=42)

# 2. Flujo completo usando los métodos del objeto
X, y = arbol.preprocesar_datos(df_clean)
arbol.dividir_datos(X, y)
arbol.entrenar()
arbol.predecir()

# 3. Evaluación y resultados
arbol.evaluar()

# 4. Consultar variables más relevantes
df_importancia = arbol.obtener_importancia_variables()
print("\n=== IMPORTANCIA DE VARIABLES ===")
print(df_importancia.to_string(index=False))

=== REPORTE DE CLASIFICACIÓN ===
              precision    recall  f1-score   support

    critical       0.75      1.00      0.86         6
    elevated       1.00      0.67      0.80         6
      normal       1.00      1.00      1.00         8

    accuracy                           0.90        20
   macro avg       0.92      0.89      0.89        20
weighted avg       0.93      0.90      0.90        20


=== MATRIZ DE CONFUSIÓN ===
                Pred: critical  Pred: elevated  Pred: normal
Real: critical               6               0             0
Real: elevated               2               4             0
Real: normal                 0               0             8

=== IMPORTANCIA DE VARIABLES ===
         Variable  Importancia (%)
       heart_rate        87.031227
      systolic_bp         6.370085
     diastolic_bp         3.940449
 body_temperature         2.658239
oxygen_saturation         0.000000
